# Notebook 02 — Graph Exploration

**Goal:** Build the skeleton graph from extracted keypoints, visualise it with NetworkX, and verify the node features look correct before training the GNN.

### Steps
1. Load saved keypoints from `data/processed/`
2. Build a PyTorch Geometric graph from one frame
3. Visualise the skeleton graph with NetworkX
4. Inspect node features (position, velocity, joint angles)
5. Check the GNN forward pass produces the right output shape

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import torch
import networkx as nx
import matplotlib.pyplot as plt

from module1.gnn.graph import (
    build_graph, build_node_features, SKELETON_EDGES,
    ANGLE_TRIPLETS, KEYPOINT_NAMES
)
from module1.pose.extractor import load_keypoints, KEYPOINT_NAMES

print('Imports OK')

In [ ]:
# ── Load keypoints saved from notebook 01 ─────────────────────────────────────
kp = load_keypoints('../data/processed/sample_clip_keypoints.npy')
print(f'Loaded keypoints shape: {kp.shape}')  # (n_frames, 33, 4)

# Use the first 50 frames as a sequence window
seq = kp[:50]   # (50, 33, 4)

In [ ]:
# ── Visualise the skeleton graph ──────────────────────────────────────────────
# Use the last frame's (x, y) coordinates as node positions
frame_kp = seq[-1]   # (33, 4)

G = nx.Graph()
for i, name in enumerate(KEYPOINT_NAMES):
    G.add_node(i, label=name)
for src, dst in SKELETON_EDGES:
    G.add_edge(src, dst)

# Use (x, -y) so it looks like a person standing upright
pos = {i: (frame_kp[i, 0], -frame_kp[i, 1]) for i in range(33)}

# Colour key joints differently
key_joints   = [23, 24, 25, 26, 27, 28, 11, 12]  # hips, knees, ankles, shoulders
node_colours = ['tomato' if i in key_joints else 'steelblue' for i in range(33)]

plt.figure(figsize=(6, 10))
nx.draw_networkx(
    G, pos=pos,
    node_color=node_colours, node_size=120,
    labels={i: KEYPOINT_NAMES[i][:6] for i in range(33)},
    font_size=6, edge_color='gray', width=1.5,
)
plt.title('Skeleton graph — red = biomechanically significant joints')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ── Inspect node features ─────────────────────────────────────────────────────
from module1.gnn.graph import build_node_features

features = build_node_features(seq)   # (33, 10)
print(f'Node features shape: {features.shape}')   # (33, 10)
print('Feature columns: [x, y, z, vx, vy, vz, ax, ay, az, joint_angle, visibility]')
print()

# Print features for key joints
for i in [23, 25, 27]:   # left hip, knee, ankle
    f = features[i]
    print(f'[{i:2d}] {KEYPOINT_NAMES[i]:<20}',
          f'pos=({f[0]:.3f}, {f[1]:.3f}, {f[2]:.3f})',
          f'vel=({f[3]:.4f}, {f[4]:.4f})',
          f'angle={f[9]:.1f}°',
          f'vis={f[10]:.2f}')

In [ ]:
# ── Build a PyG graph and run a GNN forward pass ──────────────────────────────
from torch_geometric.data import Batch
from module1.gnn.graph import build_graph
from module1.gnn.model import SkeletonGAT

graph = build_graph(seq)
print(f'Graph nodes     : {graph.x.shape}')          # (33, 10)
print(f'Graph edges     : {graph.edge_index.shape}')  # (2, n_edges)

# Run through the GNN
model = SkeletonGAT(in_channels=10, hidden_dim=64, out_dim=64, heads=4)
model.eval()

batch = Batch.from_data_list([graph])
with torch.no_grad():
    spatial_vec = model(batch.x, batch.edge_index, batch.batch)

print(f'GNN spatial vec : {spatial_vec.shape}')  # (1, 64) — ready for Transformer

In [ ]:
# ── Experiment area: try a different number of GAT layers ─────────────────────
# Modify here freely. Once you find a good config, update configs/module1.yaml

# Example: deeper model with more heads
experimental_model = SkeletonGAT(
    in_channels=10,
    hidden_dim=128,   # wider
    out_dim=64,
    heads=8,          # more heads
    dropout=0.2,
)
experimental_model.eval()

with torch.no_grad():
    out = experimental_model(batch.x, batch.edge_index, batch.batch)

print(f'Experimental model output: {out.shape}')  # still (1, 64)
total_params = sum(p.numel() for p in experimental_model.parameters())
print(f'Total parameters: {total_params:,}')